## Introduction

The folder ecom contains data about ecommerece purchases. This data is organized in folders and files so that each category is represented as a folder and files in the folder contain data related to different purchases of different brands of that category
```
i.e:
category_1/brand_a.csv
category_1/brand_b.csv
category_1/brand_c.csv
category_2/brand_a.csv
category_2/brand_b.csv
```
Write `shell script` code to do the following:

## Question 1

- Combine all data in one file
- Find the category `(product_id_code)` that has the highest sale amount

In [ ]:
cat ecom/*/*.csv > all_data.csv
# productid with highest total sales
awk -F',' 'NR>1{sum[$3]+=$4} END{for(p in sum) print sum[p],p}' all_data.csv | sort -nr | head -1 | awk '{print $2}'

1515966223509088628


In [3]:
# Combine all data 
combined="all_data.csv"
echo "event_time,order_id,product_id,price,user_id,category,brand" > "$combined"

for f in ecom/*/*.csv; 
do
  category=$(basename "$(dirname "$f")")
  brand=$(basename "$f" .csv)
  tail -n +2 "$f" | awk -F',' -v cat="$category" -v br="$brand" 'BEGIN{OFS=","} {print $1,$2,$3,$4,$5,cat,br}' >> "$combined"
done

# Product_id
awk -F',' 'NR>1{sum[$3]+=$4} END{for(p in sum) print sum[p],p}' "$combined" \
  | sort -nr | head -1 | awk '{print $2}'

1515966223509088628


In [10]:
# Combine all data (skip duplicate headers)
head -n 1 $(ls ecom/*/*.csv | head -n 1) > all_data.csv # first file of the list from ls is used and we get the first row of that
for f in ecom/*/*.csv; do
  tail -n +2 "$f" >> all_data.csv 
done

# Find product_id with highest total sales
awk -F',' 'NR>1{sum[$3]+=$4} END{for(p in sum) print sum[p],p}' all_data.csv | sort -nr | head -1 | awk '{print $2}'
# sum[$3]+=$4: add price ($4) to the total for each product_id ($3).
# NR>1: skip header row.
# END: execute after all rows are processed.
# sort -nr: sort numerically in reverse order (highest first).
# head -1: get the top entry.
# awk '{print $2}': extract product_id from the output.
# Find product_id with highest total sales
# END{for(p in sum) print sum[p],p}: print total sales and product_id for each product.


1515966223509088628


## Question 2

- Find the daily peak hour

In [ ]:
awk -F',' 'NR>1{
  split($1,a," "); #split($1,a," "): Split the first field ($1, event_time) by space into array a.
  date=a[1]; # date is the first part of the event_time (before the space).
  hour=substr(a[2],1,2); # hour is the first two characters of the time part (after the space).
  cnt[date","hour]++
}
END{for(k in cnt) print k","cnt[k]}' all_data.csv \
| sort -t',' -k1,1 -k3,3nr \
| awk -F',' '!seen[$1]++'

1970-01-01,00,821
2020-01-05,11,59
2020-01-06,11,45
2020-01-07,11,69
2020-01-08,11,45
2020-01-09,10,49
2020-01-10,13,48
2020-01-11,11,71
2020-01-12,09,86
2020-01-13,10,39
2020-01-14,12,41
2020-01-15,13,43
2020-01-16,10,30
2020-01-17,12,48
2020-01-18,11,69
2020-01-19,09,66
2020-01-20,13,42
2020-01-21,12,32
2020-01-22,07,36
2020-01-23,07,29
2020-01-24,14,44
2020-01-25,10,79
2020-01-26,12,70
2020-01-27,12,27
2020-01-28,11,40
2020-01-29,09,31
2020-01-30,09,35
2020-01-31,12,51
2020-02-01,10,79
2020-02-02,06,67
2020-02-03,12,46
2020-02-04,12,44
2020-02-05,09,39
2020-02-06,13,60
2020-02-07,08,50
2020-02-08,11,88
2020-02-09,09,88
2020-02-10,13,37
2020-02-11,09,45
2020-02-12,13,31
2020-02-13,13,58
2020-02-14,12,88
2020-02-15,10,72
2020-02-16,11,77
2020-02-17,13,37
2020-02-18,08,35
2020-02-19,13,38
2020-02-20,07,32
2020-02-21,12,44
2020-02-22,11,77
2020-02-23,11,75
2020-02-24,13,33
2020-02-25,13,40
2020-02-26,13,33
2020-02-27,11,30
2020-02-28,12,68
2020-02-29,12,72
2020-03-01,12,85
2020-03-02,09

## Question 3

- Create a separate file for each brand (the file should include all rows of that brand). Save new files in folder called "brandsData"

In [ ]:
mkdir -p brandsData

for f in ecom/*/*.csv; do
  brand=$(basename "$f" .csv) # no csv extension
  out="brandsData/${brand}.csv" # has csv extension
  if [ ! -f "$out" ]; then
    head -n 1 "$f" > "$out"
  fi
  tail -n +2 "$f" >> "$out"
done
# if [ ! -f "$out" ]; then:
#Checks if the output file for this brand ($out, e.g., sony.csv) does not exist.

#head -n 1 "$f" > "$out":
#If it does not exist, copy the header line (first line) from the current input file ($f) to the output file.
#This ensures the output file starts with the correct column headers.

# fi:
# Ends the if block.

# tail -n +2 "$f" >> "$out":
# Appends all lines except the header (from line 2 onward) from the current input file to the output file.
# This adds the data rows for the brand, without duplicating headers.


## Question 4

- Find the ids of top 5 loyal users (the use who did most transactions)

In [ ]:
awk -F',' 'NR>1{cnt[$5]++} END{for(u in cnt) print cnt[u],u}' all_data.csv \
  | sort -nr | head -5

# 1. awk -F',' 'NR>1{cnt[$5]++} END{for(u in cnt) print cnt[u],u}' all_data.csv
# -F',': Use comma as the field sepwwarator (CSV).
# NR>1: Skip the header row.
# cnt[$5]++: For each row, increment the count for user ID ($5).
# END{for(u in cnt) print cnt[u],u}: After all lines, print the count and user ID for each user.


351 1.5159156255120845e+18
346 1.5159156255128172e+18
346 1.5159156255124227e+18
345 1.5159156255121178e+18
344 1.5159156255124221e+18


## Question 5

Data is now organized in folders and files so that the folder is the category and the file contains data related to a brand
```
i.e:
category_1/brand_a.csv
category_1/brand_b.csv
category_1/brand_c.csv
category_2/brand_a.csv
category_2/brand_b.csv
..................
```

Reorganize the data so that folders represent brands and files contain dtata related to a category
```
i.e: 
brand_a/category_1.csv
brand_a/category_2.csv
brand_b/category_1.csv
brand_b/category_1.csv
brand_c/category_1.csv
..................
```
save all new data in a folder called `newEcom`

In [8]:
mkdir -p newEcom

for f in ecom/*/*.csv; do
  category=$(basename "$(dirname "$f")")
  brand=$(basename "$f" .csv)
  mkdir -p "newEcom/$brand"
  cp "$f" "newEcom/$brand/${category}.csv"
done

In [ ]:
mkdir -p newEcom

for f in econ/*/*.csv; do
    category=$(basename "$(dirname "$f")")
    brand=